# Exploratory seven-gene bulk ssGSEA

This notebook preserves the source analysis cells in their original order. It has been prepared for code sharing by standardizing paths and English comments, removing saved outputs and execution history, and disabling explicitly identified historical failures.

Run the first code cell before selecting later analysis sections. Input paths are configured by `repo_config.py`; exports are written to `results/01_bulk_seven_gene_ssgsea_exploratory/` by default. Install the repository dependencies before opening the notebook.

Each original source cell has a stable ID `source-cell-NNN`. Because this guide is a new first cell, original source cell N is cell N+1 in this release. See `cell_provenance.csv` and `README.md` for changes and inputs.

Status: exploratory historical attempt. The saved source outputs recorded gene-set filtering failures for every cohort reaching ssGSEA; no successful result from this notebook was established. The seven-gene set and original ssGSEA parameters are preserved, including the unspecified min_size. This notebook must not be cited as the generator of the existing R mean-Z-score correlation figures.


In [ ]:
# Shared repository paths; supports a kernel started from the repository root or notebooks/.
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "repo_config.py").is_file():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from repo_config import DATA_DIR, RESULTS_DIR
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "01_bulk_seven_gene_ssgsea_exploratory"
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import os

# Root directory containing bulk expression datasets
root_dir = DATA_DIR / "bulk_expression"  # Configured through the shared repository data directory

# Recursively list all files and subdirectories
for dir_path, sub_dirs, files in os.walk(root_dir):
    print(f"📂 Current directory: {dir_path}")
    
    # Subdirectories
    for sub_dir in sub_dirs:
        print(f"  ├─ Directory: {sub_dir}")
        
    # File
    for file in files:
        print(f"  ├─ File: {file}")
        
    print("-" * 50)

In [ ]:
# Install the dependencies from the repository requirements.txt before running.


In [ ]:
import os
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# --- 1. Configuration ---
base_path = DATA_DIR / "bulk_expression"
target_gene = "SRSF1"
# Define the seven-gene adenosine metabolism set
gene_sets = {
    "Adenosine_Metabolism": ["NT5E", "ENTPD1", "ADA", "ADK", "PNP", "AK1", "AK2"]
}

# Find all dataset directories whose names start with GSE
gse_folders = [f for f in os.listdir(base_path) if f.startswith("GSE") and os.path.isdir(os.path.join(base_path, f))]

results_summary = []

# --- 2. Process datasets ---
for gse in gse_folders:
    print(f"Processing dataset: {gse}")
    folder_path = os.path.join(base_path, gse)
    matrix_file = os.path.join(folder_path, f"{gse}_matrix.txt")
    
    if not os.path.exists(matrix_file):
        continue

    try:
        # Read the expression matrix (gene rows and sample columns)
        # Use \t for tab-separated data and , for comma-separated data
        df = pd.read_csv(matrix_file, sep='\t', index_col=0)

        # Check whether the target gene is present
        if target_gene not in df.index:
            print(f"  [Skip] {gse} does not contain gene {target_gene}")
            continue

        # --- 3. Calculate ssGSEA ---
        # GSEApy ssGSEA expects a DataFrame with gene rows and sample columns
        # outdir=None disables intermediate output files
        ss = gp.ssgsea(data=df,
                       gene_sets=gene_sets,
                       outdir=None,
                       sample_norm_method='rank', # ssGSEA sample normalization method
                       permutation_num=0) # Permutation testing is disabled

        # Extract scores from ss.res2d
        score_df = ss.res2d.loc["Adenosine_Metabolism"]
        
        # Extract target gene expression
        gene_exp = df.loc[target_gene]

        # Merge data
        analysis_df = pd.DataFrame({
            "SRSF1_Exp": gene_exp,
            "Adenosine_Score": score_df
        }).dropna()

        # --- 4. Correlation analysis ---
        corr, p_val = spearmanr(analysis_df["SRSF1_Exp"], analysis_df["Adenosine_Score"])
        
        results_summary.append({
            "GSE": gse,
            "Spearman_R": corr,
            "P_Value": p_val
        })

        # --- 5. Plotting ---
        plt.figure(figsize=(6, 5))
        sns.regplot(data=analysis_df, x="SRSF1_Exp", y="Adenosine_Score", 
                    scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
        plt.title(f"{gse}: SRSF1 vs Adenosine\nR={corr:.3f}, P={p_val:.3e}")
        plt.xlabel(f"{target_gene} Expression")
        plt.ylabel("Adenosine Metabolism Score (ssGSEA)")
        
        # Save figure
        plt.savefig(NOTEBOOK_RESULTS_DIR / f"{gse}_correlation_python.pdf")
        plt.close()

    except Exception as e:
        print(f"  [Error] Failed to process {gse}: {e}")

# --- 3. Summary output ---
summary_df = pd.DataFrame(results_summary)
summary_df.to_csv(NOTEBOOK_RESULTS_DIR / "Correlation_Summary_Python.csv", index=False)
print("\nDataset processing finished. Review the summary and any errors above.")